# 10 — Model Comparison & Final Model Selection
Train all 7 models, compare F1 scores, save results to CSV for Streamlit app.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from scipy.stats import randint
import matplotlib.pyplot as plt
from src.preprocess import build_preprocessor
from src.config import NUMERICAL_FEATURES, CATEGORICAL_FEATURES, RANDOM_STATE, RESULTS_PATH
import os

In [ ]:
X_train = pd.read_csv('data/processed/X_train.csv')
X_test = pd.read_csv('data/processed/X_test.csv')
y_train = pd.read_csv('data/processed/y_train.csv').values.ravel()
y_test = pd.read_csv('data/processed/y_test.csv').values.ravel()
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## Train All 7 Models

In [ ]:
results = []

def evaluate(name, pipeline, X_tr, y_tr, X_te, y_te):
    pipeline.fit(X_tr, y_tr)
    y_pred = pipeline.predict(X_te)
    scores = {
        'Model': name,
        'Accuracy': accuracy_score(y_te, y_pred),
        'Precision (class 1)': precision_score(y_te, y_pred, pos_label=1, zero_division=0),
        'Recall (class 1)': recall_score(y_te, y_pred, pos_label=1, zero_division=0),
        'Test F1 (class 1)': f1_score(y_te, y_pred, pos_label=1, zero_division=0),
    }
    results.append(scores)
    print(f"{name:30s} | F1={scores['Test F1 (class 1)']:.4f}")
    return pipeline

# 1. Dummy Baseline
evaluate('Dummy Baseline',
    Pipeline([('preprocessor', build_preprocessor()),
              ('classifier', DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE))]),
    X_train, y_train, X_test, y_test)

# 2. Logistic Regression
evaluate('Logistic Regression',
    Pipeline([('preprocessor', build_preprocessor()),
              ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))]),
    X_train, y_train, X_test, y_test)

# 3. LR + class_weight
evaluate('LR + class_weight',
    Pipeline([('preprocessor', build_preprocessor()),
              ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'))]),
    X_train, y_train, X_test, y_test)

# 4. LR + SMOTE
evaluate('LR + SMOTE',
    ImbPipeline([('preprocessor', build_preprocessor()),
                 ('smote', SMOTE(random_state=RANDOM_STATE)),
                 ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))]),
    X_train, y_train, X_test, y_test)

# 5. KNN (sampled)
SAMPLE_SIZE = 30000
np.random.seed(RANDOM_STATE)
sample_idx = np.random.choice(len(X_train), size=min(SAMPLE_SIZE, len(X_train)), replace=False)
X_train_knn = X_train.iloc[sample_idx]
y_train_knn = y_train[sample_idx]

knn_num = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", MinMaxScaler())])
knn_cat = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
knn_pre = ColumnTransformer([("num", knn_num, NUMERICAL_FEATURES), ("cat", knn_cat, CATEGORICAL_FEATURES)])
knn_pipe = Pipeline([('preprocessor', knn_pre), ('classifier', KNeighborsClassifier())])
grid_knn = GridSearchCV(knn_pipe, {'classifier__n_neighbors': [3, 5, 7, 11, 15]}, cv=5, scoring='f1', n_jobs=-1)
grid_knn.fit(X_train_knn, y_train_knn)
evaluate(f'KNN (k={grid_knn.best_params_["classifier__n_neighbors"]})',
    grid_knn.best_estimator_, X_train_knn, y_train_knn, X_test, y_test)

# 6. Decision Tree (GridSearch)
dt_pipe = Pipeline([('preprocessor', build_preprocessor()),
                     ('classifier', DecisionTreeClassifier(random_state=RANDOM_STATE))])
grid_dt = GridSearchCV(dt_pipe, {'classifier__max_depth': [3, 5, 7, 10]}, cv=5, scoring='f1', n_jobs=-1)
grid_dt.fit(X_train, y_train)
evaluate(f'DT GridSearch (depth={grid_dt.best_params_["classifier__max_depth"]})',
    grid_dt.best_estimator_, X_train, y_train, X_test, y_test)

# 7. Decision Tree (RandomizedSearch)
random_dt = RandomizedSearchCV(dt_pipe, 
    {'classifier__max_depth': [3, 5, 7, 10, None], 'classifier__min_samples_split': randint(2, 20), 'classifier__min_samples_leaf': randint(1, 10)},
    n_iter=20, cv=5, scoring='f1', random_state=RANDOM_STATE, n_jobs=-1)
random_dt.fit(X_train, y_train)
evaluate('DT RandomizedSearch',
    random_dt.best_estimator_, X_train, y_train, X_test, y_test)

## Results Table

In [ ]:
results_df = pd.DataFrame(results).round(4)
results_df = results_df.sort_values('Test F1 (class 1)', ascending=False)
print(results_df.to_string(index=False))

## Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#e74c3c' if i == 0 else '#3498db' for i in range(len(results_df))]
ax.barh(results_df['Model'], results_df['Test F1 (class 1)'], color=colors, edgecolor='black')
ax.set_xlabel('F1 Score (class 1)')
ax.set_title('Model Comparison — Test F1 on Default Class', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(True, axis='x', alpha=0.3)
plt.savefig('docs/eda_plots/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Save Results for Streamlit App

In [ ]:
results_df.to_csv(RESULTS_PATH, index=False)
print(f"Results saved to {RESULTS_PATH}")
print("The Streamlit app will load this CSV for the model comparison tab.")

## Final Model Selection
Select the model with highest Test F1 (class 1).
Store the pipeline variable for pickling in the next notebook.

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_f1 = results_df.iloc[0]['Test F1 (class 1)']
print(f"Best model: {best_model_name}")
print(f"Test F1 (class 1): {best_f1:.4f}")

# Store the best pipeline for Milestone 18
# In practice, retrain the best model config on full train set
final_pipeline = grid_dt.best_estimator_  # update this based on actual results
print(f"\nfinal_pipeline ready for pickling")